# Distributed Training: FSDP, DeepSpeed & Scaling Laws

**Module 3 - Notebook 1** | Skillsoft Track 4: ML Architect

## Objectives
- Understand data/model/pipeline parallelism
- Implement FSDP (Fully Sharded Data Parallel)
- Configure DeepSpeed ZeRO stages
- Analyze scaling efficiency

In [ ]:
# Setup - Check GPU availability
import torch
import os

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# Note: This notebook demonstrates concepts. Actual distributed training requires:
# - Multiple GPUs (or multi-node setup)
# - torchrun or deepspeed launcher
# - Proper NCCL/Gloo backend configuration

## 1. Distributed Training Concepts

In [ ]:
# Conceptual overview of parallelism strategies
parallelism_overview = """
PARALLELISM STRATEGIES:

1. DATA PARALLELISM (DP/DDP/FSDP)
   - Replicate model on each GPU
   - Split batch across GPUs
   - All-reduce gradients
   - Best for: models fitting in single GPU memory

2. MODEL PARALLELISM (Tensor/Sequence/Pipeline)
   - Split model layers across GPUs
   - Tensor Parallelism: split individual layers (megatron-lm)
   - Pipeline Parallelism: split model into stages (GPipe, PipeDream)
   - Best for: models too large for single GPU

3. ZERO REDUNDANCY OPTIMIZER (ZeRO/DeepSpeed)
   - Stage 1: Partition optimizer states
   - Stage 2: + Partition gradients
   - Stage 3: + Partition model parameters (like FSDP)
   - Offload to CPU/NVMe for massive models

4. FSDP (Fully Sharded Data Parallel)
   - PyTorch native, replaces DDP
   - Shards params, gradients, optimizer states
   - Supports activation checkpointing
   - Hybrid sharding (shard within node, replicate across nodes)
"""

print(parallelism_overview)

## 2. FSDP Implementation

In [ ]:
# fsdp_train.py
fsdp_code = """
"""FSDP Distributed Training Example."""

import torch
import torch.nn as nn
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp.fully_sharded_data_parallel import (
    CPUOffload,
    BackwardPrefetch,
)
from torch.distributed.fsdp.wrap import (
    size_based_auto_wrap_policy,
    transformer_auto_wrap_policy,
)
from torch.utils.data import DataLoader, DistributedSampler
from torchvision import datasets, transforms
import os
import argparse

def setup(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def cleanup():
    dist.destroy_process_group()

class SimpleModel(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(128, num_classes)
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

def train(rank, world_size, args):
    setup(rank, world_size)
    
    # Model
    model = SimpleModel().to(rank)
    
    # FSDP wrap policy
    auto_wrap_policy = size_based_auto_wrap_policy(min_num_params=10000)
    
    model = FSDP(
        model,
        auto_wrap_policy=auto_wrap_policy,
        cpu_offload=CPUOffload(offload_params=True) if args.cpu_offload else None,
        backward_prefetch=BackwardPrefetch.BACKWARD_PRE,
        device_id=rank,
    )
    
    # Data
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])
    dataset = datasets.FakeData(size=10000, image_size=(3, 32, 32), transform=transform)
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
    loader = DataLoader(dataset, batch_size=args.batch_size, sampler=sampler, num_workers=2)
    
    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    for epoch in range(args.epochs):
        sampler.set_epoch(epoch)
        epoch_loss = 0
        for batch_idx, (data, target) in enumerate(loader):
            data, target = data.to(rank), target.to(rank)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        if rank == 0:
            print(f"Epoch {epoch}: Loss = {epoch_loss/len(loader):.4f}")
    
    cleanup()

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--cpu-offload", action="store_true")
    args = parser.parse_args()
    
    world_size = torch.cuda.device_count()
    mp.spawn(train, args=(world_size, args), nprocs=world_size, join=True)
"""

with open("fsdp_train.py", "w") as f:
    f.write(fsdp_code)

print("fsdp_train.py created")
print("Run with: torchrun --nproc_per_node=2 fsdp_train.py")

## 3. DeepSpeed Configuration

In [ ]:
# deepspeed_config.json
ds_config = {
    "train_batch_size": 64,
    "train_micro_batch_size_per_gpu": 16,
    "gradient_accumulation_steps": 1,
    "optimizer": {
        "type": "AdamW",
        "params": {
            "lr": 1e-3,
            "weight_decay": 0.01,
            "betas": [0.9, 0.999],
            "eps": 1e-8
        }
    },
    "scheduler": {
        "type": "WarmupLR",
        "params": {
            "warmup_min_lr": 1e-5,
            "warmup_max_lr": 1e-3,
            "warmup_num_steps": 100
        }
    },
    "zero_optimization": {
        "stage": 2,
        "offload_optimizer": {"device": "cpu"},
        "offload_param": {"device": "cpu"},
        "allgather_partitions": True,
        "allgather_bucket_size": 2e8,
        "reduce_scatter": True,
        "reduce_bucket_size": 2e8,
        "overlap_comm": True
    },
    "fp16": {
        "enabled": True,
        "loss_scale": 0,
        "initial_scale_power": 16
    },
    "gradient_clipping": 1.0,
    "wall_clock_breakdown": False
}

import json
with open("deepspeed_config.json", "w") as f:
    json.dump(ds_config, f, indent=2)

print("deepspeed_config.json created (ZeRO Stage 2 with CPU offload)")
print("Stages: 1=optimizer states, 2=+gradients, 3=+params (like FSDP)")

## 4. DeepSpeed Training Script

In [ ]:
# deepspeed_train.py
ds_train_code = """
"""DeepSpeed Training Example."""

import torch
import torch.nn as nn
import deepspeed
from torch.utils.data import DataLoader, DistributedSampler
from torchvision import datasets, transforms
import argparse

class SimpleModel(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3*32*32, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.net(x)

def main():
    parser = argparse.ArgumentParser()
    parser = deepspeed.add_config_arguments(parser)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--local_rank", type=int, default=-1)
    args = parser.parse_args()
    
    model = SimpleModel()
    
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])
    dataset = datasets.FakeData(size=10000, image_size=(3, 32, 32), transform=transform)
    
    # DeepSpeed handles distributed sampler
    model_engine, optimizer, trainloader, _ = deepspeed.initialize(
        args=args,
        model=model,
        model_parameters=model.parameters(),
        training_data=dataset,
    )
    
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(args.epochs):
        model_engine.train()
        epoch_loss = 0
        for batch_idx, (data, target) in enumerate(trainloader):
            data, target = data.to(model_engine.device), target.to(model_engine.device)
            output = model_engine(data)
            loss = criterion(output, target)
            
            model_engine.backward(loss)
            model_engine.step()
            epoch_loss += loss.item()
        
        if model_engine.global_rank == 0:
            print(f"Epoch {epoch}: Loss = {epoch_loss/len(trainloader):.4f}")

if __name__ == "__main__":
    main()
"""

with open("deepspeed_train.py", "w") as f:
    f.write(ds_train_code)

print("deepspeed_train.py created")
print("Run with: deepspeed --num_gpus=2 deepspeed_train.py --deepspeed_config deepspeed_config.json")

## 5. Scaling Laws Analysis

In [ ]:
# scaling_analysis.py
scaling_code = """
"""Scaling laws analysis for distributed training."""

import matplotlib.pyplot as plt
import numpy as np

# Theoretical scaling models
def amdahls_law(p, n):
    """Amdahl's law: speedup = 1 / ((1-p) + p/n)"""
    return 1 / ((1 - p) + p / n)

def gustafsons_law(p, n):
    """Gustafson's law: speedup = n - p*(n-1)"""
    return n - p * (n - 1)

# Plot scaling curves
gpus = np.array([1, 2, 4, 8, 16, 32, 64])
parallel_fractions = [0.95, 0.99, 0.999]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for p in parallel_fractions:
    speedup = [amdahls_law(p, n) for n in gpus]
    axes[0].plot(gpus, speedup, 'o-', label=f'p={p}')
axes[0].plot(gpus, gpus, 'k--', label='Ideal')
axes[0].set_xlabel('Number of GPUs')
axes[0].set_ylabel('Speedup')
axes[0].set_title('Amdahl\'s Law (Strong Scaling)')
axes[0].legend()
axes[0].grid(True)

for p in parallel_fractions:
    speedup = [gustafsons_law(p, n) for n in gpus]
    axes[1].plot(gpus, speedup, 'o-', label=f'p={p}')
axes[1].plot(gpus, gpus, 'k--', label='Ideal')
axes[1].set_xlabel('Number of GPUs')
axes[1].set_ylabel('Scaled Speedup')
axes[1].set_title('Gustafson\'s Law (Weak Scaling)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('scaling_laws.png', dpi=150)
plt.show()

# Communication overhead model
def communication_overhead(model_size_mb, gpus, bandwidth_gbps=50):
    """Estimate all-reduce time for gradient synchronization."""
    # Ring all-reduce: 2*(n-1)/n * model_size / bandwidth
    time_sec = 2 * (gpus - 1) / gpus * model_size_mb / 1024 / bandwidth_gbps
    return time_sec

model_sizes = [100, 500, 1000, 5000]  # MB
print("\nCommunication overhead (seconds per all-reduce):")
for size in model_sizes:
    for g in [2, 4, 8, 16]:
        t = communication_overhead(size, g)
        print(f"  {size}MB model, {g} GPUs: {t*1000:.1f}ms")
"""

with open("scaling_analysis.py", "w") as f:
    f.write(scaling_code)

print("scaling_analysis.py created")

In [ ]:
# Run scaling analysis
exec(open("scaling_analysis.py").read())